# 2. Запуск и передача ответов
Порядок: REVIEW training + REVIEW exam → BBOX training + BBOX exam → проверка BBOX exam → допуск → BBOX pilot → REVIEW pilot → BBOX main → REVIEW main. Качественных исполнителей можно добавлять навыками без пересоздания основного пула. Каждый запуск — осознанная платная выдача.

In [ ]:
from pathlib import Path
import sys, os, getpass, importlib
ROOT = Path.cwd()
if ROOT.name == 'clean_annotation':
    ROOT = ROOT.parent
CLEAN = ROOT / 'clean_annotation'
if not (CLEAN / 'workflow.py').exists():
    raise RuntimeError('Откройте ноутбук из корня проекта или clean_annotation')
if str(CLEAN) not in sys.path:
    sys.path.insert(0, str(CLEAN))
import workflow as w
importlib.reload(w)
PROFILE = 'existing'  # existing: текущие проекты; new: независимое воспроизведение
ENABLE_WRITES = False  # явное включение создания/изменений; запуск заданий отдельными ячейками
TOKEN = os.environ.get('YANDEX_TASKS_TOKEN') or getpass.getpass('OAuth-токен (скрытый ввод): ')
api = w.Client(TOKEN, write=ENABLE_WRITES)
del TOKEN


## Запуск одного этапа
Обучение экзамена открывается вместе с ним. Пул отправляется на модерацию; при WAITING_MODERATION повторите после одобрения. Прошедшие REVIEW получают навык автоматически по 10 эталонам. Доступ к рабочему REVIEW: экзамен от 80%, контроль от 75% либо отсутствует. Низкое качество контролей ограничивает доступ.

In [ ]:
RUN_STAGE = False
STAGE = 'review_exam'
if RUN_STAGE:
    display(w.launch(api, PROFILE, STAGE))

## Получить ответы для просмотра и анализа
Снимки содержат задачи, исходные ответы, контрольные эталоны и настройки на момент скачивания. Ошибка скачивания не трактуется как пустой пул.

In [ ]:
DOWNLOAD = False
if DOWNLOAD:
    snapshot_folder = w.snapshot(api, PROFILE)
    print(snapshot_folder)

## Передать BBOX в независимый REVIEW
Пары: bbox_exam→review_bbox_exam, bbox_pilot→review_pilot, bbox_main→review_main, bbox_rework→review_main. Контрольные задания были загружены при настройке новых проектов. Содержимое рамок не исправляется автоматически. Технические ответы сохраняются отдельно. Проверки новой версии имеют новый annotation_id и не заменяют историю старого ответа. Передача выполняется при запуске ячейки, фонового обработчика нет.

In [ ]:
SEND = False
SOURCE, TARGET = 'bbox_exam', 'review_bbox_exam'
if SEND:
    display(w.handoff(api, PROFILE, SOURCE, TARGET))

## Итоги экзамена BBOX
Нужны 10 разных фото и не менее трёх независимых голосов для каждого. Большинство OK определяет правильный ответ; порог — 90%. UNKNOWN/технические ответы не дают автоматического допуска. Голоса проверяются по связи с исходным заданием и геометрии; ханипоты не входят в BBOX-оценку. Для ручного разбора используйте исходные снимки и документируйте решение.

In [ ]:
if 'snapshot_folder' in globals():
    exam_scores = w.grade_exam(snapshot_folder)
    display(exam_scores)

## Выдать допуск прошедшим
Отдельное действие от оплаты! source=review требует вычисленный проходной score; source=manual — ваше явное решение с сохранением основания. Не устанавливайте навык по одному факту оплаты: ACCEPTED мог появиться автоматически по истечении срока.

In [ ]:
GRANT = False
if GRANT:
    scores = {r['worker_id']:r['score'] for r in exam_scores if r['pass']}
    display(w.admit(api, PROFILE, list(scores), source='review', scores=scores))

## Открыть пилот или основную разметку
Сначала убедитесь, что есть допущенные исполнители, баланс и доступные задания. REVIEW-пул может быть открыт при очередном добавлении ответов. Не ждите окончания суток при отложенной приёмке.

In [ ]:
OPEN_WORK = False
WORK_STAGE = 'bbox_pilot'
if OPEN_WORK:
    display(w.launch(api, PROFILE, WORK_STAGE))